> **Demo only:** this notebook uses synthetic example scans and existing checkpoints. It does not train or evaluate on a held-out patient cohort. Saved probabilities and visualizations are illustrative, not accuracy evidence or clinical guidance.

# Breast DCE-MRI pCR Prediction: End-to-End Deep Learning Pipeline Demo

This notebook demonstrates the complete deep learning pipeline for predicting pathological Complete Response (pCR) from Breast DCE-MRI using **100% Deep Learning**:

1. **SimpleITK Preprocessing** — Load T0/T1 NIfTI volumes -> N4 bias correction -> isotropic resampling -> subtraction volume I_sub
2. **MONAI 3D UNet Segmentation** — Sliding-window inference -> binary tumor segmentation mask
3. **Volumetric Feature Profiling** — Quantitative 3D morphological and texture profiling
4. **3D Deep Learning Classification** — Volumetric Deep Neural Network (3D CNN / DenseNet / ResNet) -> predict pCR response probability directly from 3D volumes

In [1]:
import sys
import logging
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import SimpleITK as sitk
import torch
import joblib

# Ensure project root is on path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s - %(message)s'
)
logger = logging.getLogger('PipelineDemo')
logger.info('Pipeline Demo Initialized')

2026-08-30 00:08:38,142 [INFO] PipelineDemo - Pipeline Demo Initialized


## Phase 1: SimpleITK Preprocessing

Load synthetic T₀ (pre-contrast) and T₁ (post-contrast) volumes, apply N4 bias field correction, resample to 1.0mm³ isotropic spacing, and compute peak subtraction volume $I_{\text{sub}} = I_{\text{post}} - I_{\text{pre}}$.

In [2]:
from src.utils.dummy_generator import create_synthetic_dce_case
from src.preprocessing.mri_processor import (
    apply_n4_bias_correction,
    compute_subtraction_volume,
    resample_to_isotropic,
)

# Generate synthetic T0, T1, and mask (mimics real I-SPY1 BreastDCEDL data)
t0_raw, t1_raw, mask_raw = create_synthetic_dce_case(
    size=(128, 128, 32),
    spacing=(0.8, 0.8, 2.5),
    tumor_radius_voxels=(10, 10, 4),
    seed=42,
)
logger.info('Synthetic T0: size=%s, spacing=%s mm', t0_raw.GetSize(), t0_raw.GetSpacing())

# N4 Bias Field Correction (shrink_factor=2 for speed)
t0_n4 = apply_n4_bias_correction(t0_raw, shrink_factor=2)
t1_n4 = apply_n4_bias_correction(t1_raw, shrink_factor=2)
logger.info('N4 bias correction complete')

# Isotropic resampling to 1.0 x 1.0 x 1.0 mm³
t0_iso = resample_to_isotropic(t0_n4, target_spacing=(1.0, 1.0, 1.0), interpolator=sitk.sitkLinear)
t1_iso = resample_to_isotropic(t1_n4, target_spacing=(1.0, 1.0, 1.0), interpolator=sitk.sitkLinear)
mask_iso = resample_to_isotropic(mask_raw, target_spacing=(1.0, 1.0, 1.0), interpolator=sitk.sitkNearestNeighbor)
logger.info('Resampled T0: size=%s, spacing=%s', t0_iso.GetSize(), t0_iso.GetSpacing())

# Peak Subtraction Volume
sub_vol = compute_subtraction_volume(t0_iso, t1_iso, clip_negative=True)
logger.info('Subtraction volume I_sub: size=%s', sub_vol.GetSize())

print('✓ Phase 1 COMPLETE: T0/T1 preprocessed, subtraction volume computed')

2026-08-30 00:08:59,085 [INFO] PipelineDemo - Synthetic T0: size=(128, 128, 32), spacing=(0.8, 0.8, 2.5) mm
2026-08-30 00:09:02,636 [INFO] PipelineDemo - N4 bias correction complete
2026-08-30 00:09:02,648 [INFO] PipelineDemo - Resampled T0: size=(102, 102, 80), spacing=(1.0, 1.0, 1.0)
2026-08-30 00:09:02,654 [INFO] PipelineDemo - Subtraction volume I_sub: size=(102, 102, 80)


✓ Phase 1 COMPLETE: T0/T1 preprocessed, subtraction volume computed


In [3]:
# Visualize mid-axial slice
arr_t0 = sitk.GetArrayFromImage(t0_iso)
arr_t1 = sitk.GetArrayFromImage(t1_iso)
arr_sub = sitk.GetArrayFromImage(sub_vol)
arr_mask = sitk.GetArrayFromImage(mask_iso).astype(bool)
mid_z = arr_t0.shape[0] // 2

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(arr_t0[mid_z], cmap='gray', origin='lower')
axes[0].set_title('T₀ Pre-contrast\n(after N4 + 1mm resampling)', fontsize=11)
axes[1].imshow(arr_t1[mid_z], cmap='gray', origin='lower')
axes[1].set_title('T₁ Post-contrast', fontsize=11)
axes[2].imshow(arr_sub[mid_z], cmap='hot', origin='lower')
axes[2].contour(arr_mask[mid_z], levels=[0.5], colors='cyan', linewidths=1.5)
axes[2].set_title('I_sub (Subtraction)\n[Cyan = Lesion Contour]', fontsize=11)
for ax in axes:
    ax.axis('off')
plt.suptitle('Phase 1: SimpleITK Preprocessing', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/pipeline_phase1_preprocessing.png', dpi=120, bbox_inches='tight')
plt.show()
print('✓ Phase 1 visualization saved')

✓ Phase 1 visualization saved


C:\Users\vaish\AppData\Local\Temp\ipykernel_18712\3891741382.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Phase 2: MONAI 3D UNet Segmentation

Load the pre-trained 3D UNet checkpoint and run sliding-window inference on the subtraction volume to generate the binary tumor segmentation mask.

In [8]:
import torch
from monai.inferers import sliding_window_inference
from src.segmentation.model import build_3d_unet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info('Inference device: %s', device)

# Build model and load checkpoint
model = build_3d_unet().to(device)
checkpoint_path = Path('../checkpoints/best_monai_3dunet.pt')
if checkpoint_path.exists():
    state = torch.load(str(checkpoint_path), map_location=device)
    # Training checkpoint wraps weights; extract model_state_dict if present
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    model.load_state_dict(state)
    logger.info('Loaded 3D UNet checkpoint from %s', checkpoint_path)
else:
    logger.warning('Checkpoint not found at %s. Using randomly initialized weights for demo.', checkpoint_path)

model.eval()

# Prepare input tensor: (B=1, C=1, H, W, D)
arr_sub_np = sitk.GetArrayFromImage(sub_vol).astype(np.float32)  # (Z, Y, X)
arr_sub_np = (arr_sub_np - arr_sub_np.mean()) / (arr_sub_np.std() + 1e-7)
img_tensor = torch.from_numpy(arr_sub_np).unsqueeze(0).unsqueeze(0).to(device)  # (1,1,Z,Y,X)

with torch.no_grad():
    logits = sliding_window_inference(
        inputs=img_tensor,
        roi_size=(96, 96, 32),
        sw_batch_size=2,
        predictor=model,
        overlap=0.25,
    )
    pred_mask = (torch.sigmoid(logits) > 0.5).squeeze().cpu().numpy()

logger.info('UNet prediction shape: %s | Foreground voxels: %d', pred_mask.shape, int(pred_mask.sum()))
print('✓ Phase 2 COMPLETE: 3D UNet sliding-window inference completed')

2026-08-30 00:09:46,176 [INFO] PipelineDemo - Inference device: cpu
2026-08-30 00:09:46,184 [INFO] src.segmentation.model - Building 3D UNet: in_channels=1, out_channels=1, channels=(16, 32, 64, 128, 256), res_units=2
2026-08-30 00:09:46,241 [INFO] src.segmentation.model - 3D UNet initialized with 4805534 trainable parameters (18.33 MB)
2026-08-30 00:09:46,302 [INFO] PipelineDemo - Loaded 3D UNet checkpoint from ..\checkpoints\best_monai_3dunet.pt
2026-08-30 00:09:46,577 [INFO] PipelineDemo - UNet prediction shape: (80, 102, 102) | Foreground voxels: 216359


✓ Phase 2 COMPLETE: 3D UNet sliding-window inference completed


In [9]:
# Visualize segmentation
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(arr_sub[mid_z], cmap='gray', origin='lower')
axes[0].set_title('I_sub Input', fontsize=11)

logits_np = torch.sigmoid(logits).squeeze().cpu().numpy()
axes[1].imshow(logits_np[mid_z], cmap='hot', vmin=0, vmax=1, origin='lower')
axes[1].set_title('UNet Sigmoid Heatmap', fontsize=11)

axes[2].imshow(arr_sub[mid_z], cmap='gray', origin='lower')
axes[2].contour(pred_mask[mid_z], levels=[0.5], colors='lime', linewidths=2)
axes[2].contour(arr_mask[mid_z], levels=[0.5], colors='cyan', linewidths=1.5, linestyles='--')
axes[2].set_title('Predicted [Green] vs GT [Cyan]', fontsize=11)

for ax in axes:
    ax.axis('off')
plt.suptitle('Phase 2: MONAI 3D UNet Tumor Segmentation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/pipeline_phase2_segmentation.png', dpi=120, bbox_inches='tight')
plt.show()
print('✓ Phase 2 visualization saved')

✓ Phase 2 visualization saved


FigureCanvasAgg is non-interactive, and thus cannot be shown


## Phase 3: PyRadiomics Feature Extraction

Extract 60+ quantitative 3D shape, first-order intensity, and texture (GLCM/GLSZM/NGTDM) radiomics features from the subtraction volume and tumor mask.

In [10]:
from src.features.radiomics_extractor import DCEmriRadiomicsExtractor
import pandas as pd

extractor = DCEmriRadiomicsExtractor(
    bin_width=25.0,
    enable_shape=True,
    enable_firstorder=True,
    enable_glcm=True,
    enable_glszm=True,
    enable_ngtdm=True,
)

features = extractor.extract_features(sub_vol, mask_iso)
logger.info('Extracted %d radiomics features', len(features))

# Display as sorted table
feat_df = pd.DataFrame([
    {'Feature': k.replace('original_', ''), 'Value': round(float(v), 5)}
    for k, v in sorted(features.items())
])
print(f'\nTotal Features Extracted: {len(features)}')
feat_df.head(20)

2026-08-30 00:10:08,622 [INFO] src.features.radiomics_extractor - PyRadiomics C-extension not detected. Using native IBSI-compliant 3D feature engine.
2026-08-30 00:10:08,950 [INFO] PipelineDemo - Extracted 61 radiomics features



Total Features Extracted: 61


,Feature,Value
0,firstorder_10Percentile,1.795051e+02
1,firstorder_90Percentile,2.439803e+02
2,firstorder_Energy,1.250037e+08
3,firstorder_Entropy,1.896950e+00
4,firstorder_InterquartileRange,1.957102e+01
5,firstorder_Kurtosis,8.407490e+00
6,firstorder_Maximum,2.673087e+02
7,firstorder_Mean,2.173728e+02
8,firstorder_MeanAbsoluteDeviation,2.108429e+01
9,firstorder_Median,2.252267e+02


In [11]:
# Feature class breakdown bar chart
classes = {'Shape': 0, 'First-Order': 0, 'GLCM': 0, 'GLSZM': 0, 'NGTDM': 0}
for k in features:
    if 'shape_' in k: classes['Shape'] += 1
    elif 'firstorder_' in k: classes['First-Order'] += 1
    elif 'glcm_' in k: classes['GLCM'] += 1
    elif 'glszm_' in k: classes['GLSZM'] += 1
    elif 'ngtdm_' in k: classes['NGTDM'] += 1

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#4285f4','#34a853','#fbbc04','#ea4335','#9c27b0']
bars = ax.bar(classes.keys(), classes.values(), color=colors, edgecolor='white', width=0.6)
ax.bar_label(bars, fontsize=11, fontweight='bold')
ax.set_ylabel('Number of Features', fontsize=12)
ax.set_title('Phase 3: Radiomics Feature Class Breakdown', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../reports/figures/pipeline_phase3_radiomics.png', dpi=120, bbox_inches='tight')
plt.show()
print('✓ Phase 3 COMPLETE:', sum(classes.values()), 'radiomics features extracted')

✓ Phase 3 COMPLETE: 61 radiomics features extracted


FigureCanvasAgg is non-interactive, and thus cannot be shown


## Phase 4: 3D Deep Learning Response Prediction (pCR)

Load the pre-trained 3D Deep Learning classifier (`checkpoints/best_3d_classifier.pt`) and perform volumetric inference on the 3D DCE-MRI volume to predict pathological Complete Response (pCR) probability.

In [ ]:
import torch
import torch.nn.functional as F
from src.models.classifier3d import build_3d_classifier

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
clf_path = Path('../checkpoints/best_3d_classifier.pt')

# Instantiate 3D Deep Learning Classifier
classifier_model = build_3d_classifier(
    spatial_dims=3,
    in_channels=1,
    num_classes=1,
    backbone='simple_cnn3d',
    dropout=0.2,
    device=device,
)

if clf_path.exists():
    checkpoint = torch.load(str(clf_path), map_location=device)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        classifier_model.load_state_dict(checkpoint['model_state_dict'])
    else:
        classifier_model.load_state_dict(checkpoint)
    logger.info('Loaded 3D Deep Learning classifier from %s', clf_path)
else:
    logger.warning('Checkpoint not found at %s. Using initialized weights.', clf_path)

classifier_model.eval()

# Prepare 3D volumetric input tensor for the deep classifier: (B=1, C=1, H, W, D)
with torch.no_grad():
    resized_input = F.interpolate(img_tensor, size=(64, 64, 24), mode='trilinear', align_corners=False)
    logits = classifier_model(resized_input)
    pcr_prob = float(torch.sigmoid(logits).squeeze().cpu().item())
    pcr_pred = int(pcr_prob >= 0.5)

pred_label_str = 'pCR (Responder)' if pcr_pred == 1 else 'non-pCR (Non-Responder)'
logger.info('3D Deep Learning pCR Probability: %.4f | Predicted Label: %s', pcr_prob, pred_label_str)
print('\n--- 3D Deep Learning Response Prediction ---')
print(f'  Input 3D Tensor Shape : {tuple(resized_input.shape)}')
print(f'  pCR Probability       : {pcr_prob:.4f}')
print(f'  Prediction            : {pred_label_str}')


In [ ]:
# Visualize 3D Deep Learning Response Prediction Confidence
fig, ax = plt.subplots(figsize=(8, 3.5))

classes = ['non-pCR (Non-Responder)', 'pCR (Responder)']
probabilities = [1.0 - pcr_prob, pcr_prob]
colors = ['#ea4335' if pcr_pred == 0 else '#bdc1c6', '#34a853' if pcr_pred == 1 else '#bdc1c6']

bars = ax.barh(classes, probabilities, color=colors, edgecolor='white', height=0.55)
for bar, val in zip(bars, probabilities):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2, f'{val:.1%}', va='center', fontsize=11, fontweight='bold')
ax.set_xlim(0, 1.2)
ax.set_xlabel('Predicted Response Probability', fontsize=12)
ax.set_title(
    f'Phase 4: 3D Deep Learning Response Prediction (pCR)\n'
    f'Predicted Class: {pred_label_str} (Confidence: {max(probabilities):.1%})',
    fontsize=13, fontweight='bold'
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/pipeline_phase4_classification.png', dpi=120, bbox_inches='tight')
plt.show()
print('✓ Phase 4 COMPLETE: 3D Deep Learning response prediction visualization saved')


## Pipeline Summary

| Phase | Module | Output |
|---|---|---|
| 1 | SimpleITK Preprocessing | Subtraction volume at 1.0mm3 isotropic spacing |
| 2 | MONAI 3D UNet | Binary tumor segmentation mask |
| 3 | Volumetric Feature Profiling | 60+ quantitative 3D features |
| 4 | 3D Deep Learning Classifier | pCR response probability prediction |

In [ ]:
print('=' * 60)
print('  BREAST DCE-MRI 3D DEEP LEARNING PIPELINE COMPLETE')
print('=' * 60)
print(f'  Features Profiled  : {len(features)}')
print(f'  3D DL pCR Prob     : {pcr_prob:.4f}')
print(f'  Final Prediction   : {pred_label_str}')
print('=' * 60)
